# Fake News Detection

In [1]:
# Importing Necessary  Libraries

import pandas as pd
import numpy as np
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")


In [2]:
# Step 1: Load & Label Data

true = pd.read_csv(r"C:\Users\Tuhin\OneDrive\Desktop\True.csv")
fake = pd.read_csv(r"C:\Users\Tuhin\OneDrive\Desktop\Fake.csv")

true['label'] = 1  # Real
fake['label'] = 0  # Fake

data = pd.concat([true, fake]).sample(frac=1).reset_index(drop=True)
data['text'] = data['title'] + " " + data['text']

In [3]:
# Step 2: Clean Text
def clean(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|<.*?>|[^a-zA-Z\s]", "", text)
    return text

data['text'] = data['text'].apply(clean)

In [4]:
# Step 3: Tokenize & Pad
max_words = 5000
max_len = 200

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(data['text'])
sequences = tokenizer.texts_to_sequences(data['text'])
X = pad_sequences(sequences, maxlen=max_len)
y = data['label'].values

In [5]:
# Step 4: Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
# Step 5: Load GloVe Embeddings
# ==========================
embedding_index = {}
with open(r'C:\Users\Tuhin\OneDrive\Desktop\glove.6B.100d.txt', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = vector

embedding_dim = 100
word_index = tokenizer.word_index
embedding_matrix = np.zeros((max_words, embedding_dim))

for word, i in word_index.items():
    if i < max_words:
        embedding_vector = embedding_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

In [7]:
# Step 6: Build LSTM Model
# ==========================
model = Sequential()
model.add(Embedding(max_words, embedding_dim, weights=[embedding_matrix], input_length=max_len, trainable=False))
model.add(LSTM(64, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │         500,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 500,000 (1.91 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 500,000 (1.91 MB)

In [8]:
# Step 7: Train Model
# ==========================
model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.1)

Epoch 1/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 223s 425ms/step - accuracy: 0.8472 - loss: 0.3386 - val_accuracy: 0.9474 - val_loss: 0.1506
Epoch 2/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 218s 430ms/step - accuracy: 0.9444 - loss: 0.1533 - val_accuracy: 0.9671 - val_loss: 0.0934
Epoch 3/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 221s 438ms/step - accuracy: 0.9621 - loss: 0.1053 - val_accuracy: 0.9788 - val_loss: 0.0618
Epoch 4/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 225s 444ms/step - accuracy: 0.9713 - loss: 0.0800 - val_accuracy: 0.9800 - val_loss: 0.0587
Epoch 5/5
506/506 ━━━━━━━━━━━━━━━━━━━━ 229s 452ms/step - accuracy: 0.9768 - loss: 0.0665 - val_accuracy: 0.9830 - val_loss: 0.0501


In [10]:
# Step 8: Evaluate
# ==========================
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)
print("Accuracy:", f'{accuracy_score(y_test, y_pred):.2f}')

281/281 ━━━━━━━━━━━━━━━━━━━━ 12s 41ms/step
Accuracy: 0.98


In [11]:
# Step 9: Test on Custom News
# ==========================
def predict_news(news):
    news = clean(news)
    seq = tokenizer.texts_to_sequences([news])
    padded = pad_sequences(seq, maxlen=max_len)
    pred = model.predict(padded)[0][0]
    return "Real" if pred >= 0.5 else "Fake"

# Example:
print(predict_news("NASA announces a new mission to Jupiter"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
Fake
